In [61]:
import sys
sys.path.append('/home/mila/l/le.zhang/scratch/DeepRerank')
from pyserini.search import get_topics, get_qrels
# from pyserini.search.lucene import LuceneSearcher
from run_evaluation import THE_TOPICS, THE_INDEX
from trec_eval import EvalFunction, trec_eval
import json
data = 'dl21'

# searcher = LuceneSearcher.from_prebuilt_index(THE_INDEX[data])
DLV2 = ['dl20', 'dl21', 'dl22', 'dl23']
topics = get_topics(THE_TOPICS[data] if data not in DLV2 else data)
qrels = get_qrels(THE_TOPICS[data])

def read_bright_topics(path):
    query_map = {}
    with open(path, 'r') as f:
        for line in f:
            qid, query = line.strip().split('\t')
            query_map[qid] = query
    return query_map

def read_bright_qrels(qrels_path):
    qrels_dict = {}
    # Use a more efficient file reading approach
    with open(qrels_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, _, docid, rel = parts[:4]
                qrels_dict.setdefault(qid, {})[docid] = int(rel)
    return qrels_dict

def read_bm25_run(run_path):
    run_dict = {}
    with open(run_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                qid, _, docid, rank, score = parts[:5]
                if qid not in run_dict:
                    run_dict[qid] = {}
                run_dict[qid][docid] = float(score)
    return run_dict

def read_corpus_from_jsonl(corpus_path):
    corpus = {}
    with open(corpus_path, 'r') as f:
        for line in f:
            doc = json.loads(line)
            docid = doc['id']
            text = doc['contents']
            corpus[docid] = text
    return corpus

def get_hits_from_run_bright(base_dir, data):
    topic_dict = read_bright_topics(f"{base_dir}/data/bright/data/pyserini_queries/{data}.tsv")
    run_dict = read_bm25_run(f"{base_dir}/data/bright/runs_bm25/bm25.{data}.filtered.trec")
    corpus = read_corpus_from_jsonl(f"{base_dir}/data/bright/data/pyserini_corpus/{data}/{data}.jsonl")
    rank_results = []
    for qid, docids in run_dict.items():
        result = {}
        result['query'] = topic_dict[qid]
        hits = []
        rank = 1
        for docid, score in docids.items():
            if docid in corpus:
                hits.append({'docid': docid, 'score': score, 'content': corpus[docid], 'qid': qid, 'rank': rank})
                rank += 1
        result['hits'] = hits
        rank_results.append(result)
    return rank_results

data = 'biology'
base_dir = "/home/mila/l/le.zhang/scratch/DeepRerank"
topic_dict =read_bright_topics(f"{base_dir}/data/bright/data/pyserini_queries/{data}.tsv")
# qrels_dict = read_bright_qrels(f"{base_dir}/data/bright/data/pyserini_qrels/{data}.tsv")
# bm25_run = read_bm25_run(f"{base_dir}/data/bright/runs_bm25/bm25.{data}.filtered.trec")
rank_results = get_hits_from_run_bright(base_dir, data)


In [62]:
len(topic_dict)

103

In [30]:
metric, _ =trec_eval(qrels_dict, bm25_run, k_values=(1, 5, 10))

In [31]:
metric

{'NDCG@1': 0.15534,
 'NDCG@5': 0.16345,
 'NDCG@10': 0.18244,
 'MAP@1': 0.05382,
 'MAP@5': 0.11096,
 'MAP@10': 0.12405,
 'Recall@1': 0.05382,
 'Recall@5': 0.16397,
 'Recall@10': 0.22394}